# LeetCode 755 — Pour Water bằng Python

Link đề: https://leetcode.com/problems/pour-water/ · Độ khó: **Medium** · Chủ đề: Array, Simulation

Skeleton LeetCode:

```python
class Solution:
    def pourWater(self, heights: List[int], volume: int, k: int) -> List[int]:
        ...
```

Ghi chú: đây là bài **Premium** (cần tài khoản trả phí để xem đề trên LeetCode). Đề trong notebook được viết theo hiểu biết về bài, nên hãy **đối chiếu lại đề gốc nếu bạn có tài khoản Premium**. Ba ví dụ ở mục 2 lấy theo đề; các ca viết tay ở mục 12 là tự dựng (đáp án đã tính tay và kiểm chứng bằng code). Đây là bài **mô phỏng (simulation)**: điểm khó không phải thuật toán cao siêu mà là **đọc đúng luật** "chảy trái – chảy phải – dâng lên" và viết đúng xử lý các mặt phẳng bằng nhau.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho bản đồ độ cao `heights`, đổ `volume` giọt nước lần lượt xuống vị trí `k`; mỗi giọt **ưu tiên chảy trái** nếu thật sự sẽ **tụt xuống thấp hơn**, nếu không thì **chảy phải** nếu tụt được, nếu không nữa thì **dâng lên tại chỗ** (`k`). Trả về bản đồ sau khi đổ hết.
2. Nắm các công cụ Python: **danh sách sửa tại chỗ** (`heights[j] += 1`) và **sao chép** (`heights[:]`), vòng `for d in (-1, 1)` để xử lý hai hướng bằng một đoạn code, **so sánh dây chuyền** `0 <= i + d < n`, mệnh đề `for ... else`, `min(..., key=...)` với khóa dạng tuple, và `while` có điều kiện hợp.
3. Đi từ **mô phỏng giọt nước lăn từng bước theo đúng luật** → **một lượt quét mỗi giọt, nhớ ô thấp hơn cuối cùng** → **chọn "ô thấp nhất, gần `k` nhất" trong đoạn đi được** → **đổ theo lô** (gom nhiều giọt cùng rơi vào một ô).
4. Hiểu mấu chốt: một giọt từ vị trí `i` đi sang `i + d` **chỉ khi** `heights[i + d] <= heights[i]` (không leo lên). Giọt chỉ **thực sự đổi chỗ** nếu trong đoạn đi được có một ô **thấp hơn hẳn** chỗ bắt đầu; điểm dừng là ô thấp nhất, và nếu nhiều ô thấp bằng nhau thì là ô **gần `k` nhất** (cuối cùng của lần tụt thấp hơn hẳn).
5. Tránh các lỗi kinh điển: dùng `<=` / `<` sai (mặt phẳng bằng nhau không phải "tụt"), dừng ngay khi gặp ô bằng nhau, chọn ô **xa** `k` thay vì ô **gần** `k`, quên ưu tiên **trái trước phải**, và quên rằng hai bên ngoài biên là **tường vô hạn** (nước không chảy ra ngoài).

## 2. Đề bài & ví dụ

Bạn được cho một bản đồ độ cao dạng mảng số nguyên `heights`, trong đó `heights[i]` là độ cao địa hình tại cột `i` (mỗi cột rộng `1`). Bạn cũng được cho hai số nguyên `volume` và `k`: `volume` đơn vị nước sẽ rơi xuống vị trí `k`, **từng đơn vị một**.

Nước rơi đầu tiên xuống cột `k` và dừng ở **đỉnh** của địa hình hoặc nước tại cột đó. Sau đó nó chảy theo luật:

1. Nếu giọt nước **cuối cùng sẽ tụt xuống** khi đi về **bên trái**, thì đi sang trái.
2. Nếu không, mà đi **bên phải** thì tụt xuống, thì đi sang phải.
3. Nếu cả hai đều không, nó **dâng lên** tại vị trí hiện tại (cột `k`).

"Cuối cùng sẽ tụt xuống" nghĩa là: nếu cứ đi theo hướng đó, giọt nước sẽ ở một **mức** thấp hơn mức hiện tại. **Mức (level)** của một cột là **độ cao địa hình + lượng nước** trong cột. Giọt nước chỉ được đi sang cột kề nếu cột đó có mức **bằng hoặc thấp hơn** (không leo lên).

Giả sử hai bên ngoài mảng là **địa hình cao vô hạn** (nước không chảy ra ngoài). Cũng không có chuyện nước chia đều lên nhiều ô: mỗi đơn vị nước nằm trọn trong **một** ô.

**Ví dụ 1**

- Input: `heights = [2,1,1,2,1,2,2]`, `volume = 4`, `k = 3`
- Output: `[2,2,2,3,2,2,2]`

**Ví dụ 2**

- Input: `heights = [1,2,3,4]`, `volume = 2`, `k = 2`
- Output: `[2,3,3,4]`
- Giải thích: giọt cuối cùng dừng ở chỉ số 1, vì đi xa hơn về bên trái sẽ không tụt xuống thấp hơn.

**Ví dụ 3**

- Input: `heights = [3,1,3]`, `volume = 5`, `k = 1`
- Output: `[4,4,4]`

**Ràng buộc:**

- `1 <= heights.length <= 100`
- `0 <= heights[i] <= 99`
- `0 <= volume <= 2000`
- `0 <= k < heights.length`

Chú ý: kết quả là **chính bản đồ sau khi đã thêm nước** (địa hình + nước), nên tổng của kết quả luôn bằng `sum(heights) + volume`.

In [1]:
from typing import List
import random
import sys

examples = [
    ([2, 1, 1, 2, 1, 2, 2], 4, 3, [2, 2, 2, 3, 2, 2, 2]),
    ([1, 2, 3, 4], 2, 2, [2, 3, 3, 4]),
    ([3, 1, 3], 5, 1, [4, 4, 4]),
]
for heights, volume, k, expected in examples:
    print(f"heights={heights} volume={volume} k={k} -> kỳ vọng {expected}")
    assert sum(expected) == sum(heights) + volume     # nước được bảo toàn

heights=[2, 1, 1, 2, 1, 2, 2] volume=4 k=3 -> kỳ vọng [2, 2, 2, 3, 2, 2, 2]
heights=[1, 2, 3, 4] volume=2 k=2 -> kỳ vọng [2, 3, 3, 4]
heights=[3, 1, 3] volume=5 k=1 -> kỳ vọng [4, 4, 4]


## 3. Kiến thức nền

### 3.1. Mức (level) và danh sách sửa tại chỗ

Ta dùng **một** danh sách `h` để lưu **mức** từng cột (địa hình + nước). Đổ một giọt vào cột `j` chỉ là `h[j] += 1`. `list` trong Python là **mutable**: gán `g = h` không tạo bản sao mà hai tên cùng trỏ vào một list; muốn bản sao độc lập phải viết `h[:]` (hoặc `list(h)`). Quan trọng khi test nhiều cách giải trên cùng đầu vào.

In [2]:
h = [2, 1, 1, 2]
alias = h               # cùng một list!
copy = h[:]             # bản sao độc lập
h[1] += 1
print("h     =", h)
print("alias =", alias, "(đổi theo h)")
print("copy  =", copy, "(không đổi)")

h     = [2, 2, 1, 2]
alias = [2, 2, 1, 2] (đổi theo h)
copy  = [2, 1, 1, 2] (không đổi)


### 3.2. Hai hướng bằng một đoạn code: `for d in (-1, 1)`

Gọi `d = -1` là đi **trái**, `d = +1` là đi **phải**. Viết `for d in (-1, 1):` để lặp qua hai hướng theo đúng thứ tự **trái trước, phải sau** — đúng ưu tiên của đề — và dùng chung một thân vòng lặp. Ô kề của `i` theo hướng `d` là `i + d`.

In [3]:
i = 3
for d in (-1, 1):
    print("hướng", "trái" if d == -1 else "phải", "-> ô kề của", i, "là", i + d)

hướng trái -> ô kề của 3 là 2
hướng phải -> ô kề của 3 là 4


### 3.3. So sánh dây chuyền và kiểm tra biên

Python cho phép `0 <= x < n` thay cho `x >= 0 and x < n`. Trong bài này điều kiện "còn trong mảng" là `0 <= i + d < n`. Ngoài mảng là **tường vô hạn**, tức nước **không** đi tiếp — nên khi `i + d` ra ngoài, vòng lặp đi bộ phải **dừng**.

In [4]:
n = 5
for x in (-1, 0, 4, 5):
    print(x, "trong mảng?", 0 <= x < n)

# điều kiện trong while: kiểm biên TRƯỚC khi đọc h[i+d] (and ngắn mạch)
h = [3, 2, 1]
i, d = 2, 1
print("còn đi được?", 0 <= i + d < len(h) and h[i + d] <= h[i])   # i + d = 3: ngoài mảng, không đọc h[3]

-1 trong mảng? False
0 trong mảng? True
4 trong mảng? True
5 trong mảng? False
còn đi được? False


### 3.4. `for ... else`

Vòng `for` có thể có nhánh `else`, nó **chỉ chạy khi vòng lặp kết thúc mà không gặp `break`**. Rất tiện cho luật "thử trái, rồi thử phải, nếu **cả hai** không được thì dâng lên": nếu một hướng thành công ta `break`, còn nếu cả hai hướng đều hỏng thì `else` chạy.

In [5]:
for d in (-1, 1):
    if d == 5:            # điều kiện không bao giờ đúng
        print("tìm thấy")
        break
else:
    print("else chạy vì không có break")

for d in (-1, 1):
    if d == 1:
        print("tìm thấy hướng", d)
        break
else:
    print("(dòng này không in)")

else chạy vì không có break
tìm thấy hướng 1


### 3.5. `min(..., key=...)` với khóa là tuple

`min(iterable, key=f)` chọn phần tử có `f(x)` nhỏ nhất. Nếu `f` trả về **tuple**, Python so sánh **từng phần** theo thứ tự: trước hết phần tử đầu, hòa mới xét phần tử sau. Ta dùng khóa `(mức, khoảng cách đến k)` để chọn ô **thấp nhất**, và nếu bằng nhau thì ô **gần `k` nhất**.

In [6]:
levels = {2: 1, 1: 1, 0: 2}              # chỉ số -> mức
k = 3
best = min(levels, key=lambda idx: (levels[idx], abs(idx - k)))
print("ô thấp nhất, gần k nhất:", best)  # 2 (mức 1) gần k=3 hơn ô 1 (mức 1)
print(sorted(levels, key=lambda idx: (levels[idx], abs(idx - k))))

ô thấp nhất, gần k nhất: 2
[2, 1, 0]


### 3.6. Vẽ bản đồ bằng chữ để "nhìn thấy" nước

Để hiểu một bài mô phỏng, hãy vẽ nó. Hàm `draw(h, base)` in từng hàng từ trên xuống: `#` là địa hình (`base`), `~` là nước (phần `h` cao hơn `base`).

In [7]:
def draw(h, base):
    top = max(h)
    for level in range(top, 0, -1):
        row = ""
        for x, b in zip(h, base):
            if level <= b:
                row += "#"
            elif level <= x:
                row += "~"
            else:
                row += "."
        print(row)
    print("-" * len(h))

base = [2, 1, 1, 2, 1, 2, 2]
print("ban đầu:")
draw(base, base)
print("kết quả ví dụ 1 ([2,2,2,3,2,2,2]):")
draw([2, 2, 2, 3, 2, 2, 2], base)

ban đầu:
#..#.##
#######
-------
kết quả ví dụ 1 ([2,2,2,3,2,2,2]):
...~...
#~~#~##
#######
-------


## 4. Ý tưởng nền tảng / trực giác

Hãy đổ **một** giọt vào ví dụ 1 và nhìn:

```
heights = [2,1,1,2,1,2,2], k = 3
```

Giọt rơi xuống cột `3` (mức 2). Đi **trái**: cột `2` mức `1` (thấp hơn, đi được), cột `1` mức `1` (bằng, đi được), cột `0` mức `2` (cao hơn, **không** đi được — dừng). Trên đường đi, giọt đã gặp mức thấp hơn (`1 < 2`) nên **sẽ tụt xuống** → chọn trái. Nó dừng ở đâu? Có hai ô mức `1` là cột `2` và cột `1`; giọt tụt xuống ô đầu tiên nó gặp — **cột `2`**, ô gần `k` nhất — rồi nằm đó. (Ô `1` sẽ được lấp tiếp ở giọt sau.)

Vậy có hai câu hỏi cần trả lời cho **mỗi giọt**:

1. Đi một hướng thì có **tụt xuống được** không? (để biết chọn hướng nào)
2. Nếu tụt, nó **dừng ở ô nào**?

Ta viết hàm `falls(h, p, d)` đúng theo luật để trả lời câu 1: bắt đầu từ `p`, cứ đi sang `p + d` khi ô đó **không cao hơn** ô đang đứng; nếu có lúc mức **thấp hơn mức xuất phát** thì trả `True`.

In [8]:
def falls(h, p, d):
    """Từ cột p, nếu cứ đi theo hướng d (chỉ sang ô không cao hơn) thì cuối cùng có xuống mức thấp hơn h[p] không?"""
    i = p
    n = len(h)
    while 0 <= i + d < n and h[i + d] <= h[i]:
        i += d
        if h[i] < h[p]:
            return True
    return False

h = [2, 1, 1, 2, 1, 2, 2]
print("từ 3 đi trái tụt?", falls(h, 3, -1))     # True
print("từ 3 đi phải tụt?", falls(h, 3, 1))      # True (cột 4 mức 1)
h2 = [2, 2, 2, 2, 1, 2, 2]
print("từ 3 đi trái tụt?", falls(h2, 3, -1))    # False: toàn mặt phẳng bằng nhau rồi chạm cột 0 và tường
print("từ 3 đi phải tụt?", falls(h2, 3, 1))     # True

từ 3 đi trái tụt? True
từ 3 đi phải tụt? True
từ 3 đi trái tụt? False
từ 3 đi phải tụt? True


Lưu ý đúng chỗ khó: **mặt phẳng bằng nhau không phải là "tụt"**. Ở `h2` đi trái là `2,2,2` rồi hết mảng — không có ô nào thấp hơn → `False`.

## 5. Cách 1 — Brute Force: mô phỏng giọt nước lăn từng bước theo đúng luật

**Ý tưởng.** Làm đúng như mô tả: giọt bắt đầu ở `p = k`. Lặp lại: nếu `falls(h, p, -1)` thì **bước một ô sang trái**; nếu không mà `falls(h, p, +1)` thì bước sang phải; nếu cả hai không thì **dừng** và dâng nước tại `p` (`h[p] += 1`). Sau mỗi bước ta **hỏi lại** luật tại vị trí mới.

Đây là cách **trung thành nhất với đề**, nên ta cũng dùng nó làm "trọng tài" để đối chiếu các cách khác (kèm thêm các ca tính tay và kiểm tra bảo toàn nước ở mục 12).

In [9]:
def pour_ball(heights, volume, k):
    h = heights[:]
    for _ in range(volume):
        p = k
        while True:
            if falls(h, p, -1):
                p -= 1
            elif falls(h, p, 1):
                p += 1
            else:
                break
        h[p] += 1
    return h

for heights, volume, k, expected in examples:
    got = pour_ball(heights, volume, k)
    assert got == expected, (got, expected)
    print(heights, volume, k, "->", got)

[2, 1, 1, 2, 1, 2, 2] 4 3 -> [2, 2, 2, 3, 2, 2, 2]
[1, 2, 3, 4] 2 2 -> [2, 3, 3, 4]
[3, 1, 3] 5 1 -> [4, 4, 4]


**Độ phức tạp.** Mỗi giọt có thể bước tối đa `n` lần, mỗi bước `falls` quét tối đa `n` ô → **O(v · n²)** thời gian (`v = volume`). **Space O(1)** thêm (ngoài bản sao `h`). Với `v = 2000`, `n = 100` có thể lên tới hàng chục triệu bước — chậm trong Python.

**Nút thắt.** Ta **tính lại nhiều lần** cùng một đoạn đường: mỗi bước lăn đều quét lại phần đường phía trước. Nhưng đường đi của một giọt rất đơn giản (một chiều, không quay đầu), nên chỉ cần **một lượt quét** là biết điểm dừng.

## 6. Ý tưởng thuật toán / bước đột phá

Với **một** hướng `d`, đường đi của giọt là một dãy ô `k, k+d, k+2d, ...` với mức **không tăng** (mỗi ô kề không cao hơn ô trước), dừng khi gặp ô cao hơn hoặc ra khỏi mảng. Gọi đoạn đi được đó là **đoạn chảy (run)**. Vì mức **không tăng**, ô **thấp nhất** nằm ở **cuối đoạn**, và:

- Giọt **tụt được** ⇔ mức thấp nhất của đoạn `<` `h[k]`.
- Giọt **dừng** ở ô thấp nhất; nếu nhiều ô cùng thấp nhất (một mặt bằng ở cuối đoạn) thì dừng ở ô **đầu tiên** của mặt bằng đó khi đi từ `k` — tức ô **gần `k` nhất**. Đó cũng chính là ô nơi **lần tụt thấp hơn hẳn cuối cùng** xảy ra trên đường đi.

Vậy một lượt quét là đủ: đi `i` từ `k` theo hướng `d` khi `h[i + d] <= h[i]`; mỗi khi `h[i + d] < h[i]` thì ghi `j = i + d`. Cuối cùng `j` là điểm dừng (nếu `j == k` nghĩa là không tụt). Hãy xem trên ví dụ 1.

In [10]:
h = [2, 1, 1, 2, 1, 2, 2]
k = 3
for d in (-1, 1):
    i = j = k
    path = [k]
    while 0 <= i + d < len(h) and h[i + d] <= h[i]:
        if h[i + d] < h[i]:
            j = i + d
        i += d
        path.append(i)
    print(f"hướng {'trái' if d == -1 else 'phải'}: đường đi {path}, mức {[h[x] for x in path]}, j = {j}")

hướng trái: đường đi [3, 2, 1], mức [2, 1, 1], j = 2
hướng phải: đường đi [3, 4], mức [2, 1], j = 4


Hướng trái: đường đi `3 → 2 → 1` (mức `2,1,1`), lần tụt thấp hơn hẳn là bước `3 → 2`, nên `j = 2` (gần `k` nhất trong hai ô mức `1`). Vì `j != k`, giọt chọn **trái** (ưu tiên trái) và dâng ở cột `2`.

## 7. Các cách tiếp theo

4 cách tổng cộng (cách 1 đã ở mục 5):

| # | Cách | Time | Space |
|---|---|---|---|
| 1 | Mô phỏng giọt lăn từng bước | O(v·n²) | O(1) |
| 2 | Một lượt quét mỗi giọt, nhớ ô tụt cuối | O(v·n) | O(1) |
| 3 | Chọn ô `min` theo khóa `(mức, khoảng cách)` trong đoạn chảy | O(v·n) | O(n) |
| 4 | Đổ theo lô (gom nhiều giọt cùng một ô) | O(n · số lô) ≤ O(v·n) | O(1) |

### 7.1. Cách 2 — Một lượt quét mỗi giọt, nhớ ô tụt cuối cùng

**Giải thích từng đoạn.**

1. Lặp `volume` lần (mỗi lần một giọt).
2. `for d in (-1, 1)`: thử **trái trước**, rồi phải.
3. `i = j = k`: `i` là con trỏ đi, `j` là ô dừng ứng viên (ban đầu chính `k`).
4. `while 0 <= i + d < n and h[i + d] <= h[i]`: còn trong mảng và không phải leo lên thì tiếp tục.
5. Trong vòng: nếu `h[i + d] < h[i]` thì `j = i + d` (một lần tụt thấp hơn hẳn); rồi `i += d`.
6. Sau khi đi xong một hướng: nếu `j != k` thì `h[j] += 1` và `break` (không thử hướng còn lại).
7. Nếu cả hai hướng không tụt (`for ... else`): `h[k] += 1`.

Cẩn thận hai chi tiết hay sai: dùng `<=` để **đi qua** mặt bằng bằng nhau, nhưng chỉ cập nhật `j` khi `<` thật sự.

In [11]:
def pour_one_pass(heights, volume, k):
    h = heights[:]
    n = len(h)
    for _ in range(volume):
        for d in (-1, 1):
            i = j = k
            while 0 <= i + d < n and h[i + d] <= h[i]:
                if h[i + d] < h[i]:
                    j = i + d
                i += d
            if j != k:
                h[j] += 1
                break
        else:
            h[k] += 1
    return h

for heights, volume, k, expected in examples:
    got = pour_one_pass(heights, volume, k)
    assert got == expected, (got, expected)
    print(heights, volume, k, "->", got)

[2, 1, 1, 2, 1, 2, 2] 4 3 -> [2, 2, 2, 3, 2, 2, 2]
[1, 2, 3, 4] 2 2 -> [2, 3, 3, 4]
[3, 1, 3] 5 1 -> [4, 4, 4]


In [12]:
# Code nộp LeetCode — Cách 2: một lượt quét mỗi giọt
class Solution:
    def pourWater(self, heights: List[int], volume: int, k: int) -> List[int]:
        n = len(heights)
        for _ in range(volume):
            for d in (-1, 1):
                i = j = k
                while 0 <= i + d < n and heights[i + d] <= heights[i]:
                    if heights[i + d] < heights[i]:
                        j = i + d
                    i += d
                if j != k:
                    heights[j] += 1
                    break
            else:
                heights[k] += 1
        return heights

for heights, volume, k, expected in examples:
    assert Solution().pourWater(list(heights), volume, k) == expected
print("Cách 2 qua ví dụ đề")

Cách 2 qua ví dụ đề


**Độ phức tạp.** Mỗi giọt quét tối đa `2n` ô → **Time O(v · n)** (`2000 × 100 × 2 = 4×10^5` — rất nhanh). **Space O(1)** (sửa tại chỗ; trong notebook ta truyền bản sao để các cách không dùng chung dữ liệu).

### 7.2. Cách 3 — Chọn ô `min` theo khóa `(mức, khoảng cách)` trong đoạn chảy

**Ý tưởng.** Cùng bản chất nhưng tách làm hai bước dễ đọc:

1. **Dựng đoạn chảy** theo hướng `d`: danh sách `run` các chỉ số `k, k+d, ...` chừng nào `h[i + d] <= h[i]`.
2. **Chọn ô dừng**: `best = min(run, key=lambda x: (h[x], abs(x - k)))` — mức thấp nhất, hòa thì gần `k` nhất. Nếu `h[best] < h[k]` thì giọt đi về hướng này và dâng ở `best`.

Hàm `settle(h, k)` trả về chỉ số mà giọt tiếp theo sẽ dừng (không sửa `h`); ta dùng lại hàm này ở cách 4.

In [13]:
def settle(h, k):
    """Chỉ số nơi giọt kế tiếp sẽ dừng (không sửa h)."""
    n = len(h)
    for d in (-1, 1):
        run = [k]
        i = k
        while 0 <= i + d < n and h[i + d] <= h[i]:
            i += d
            run.append(i)
        best = min(run, key=lambda x: (h[x], abs(x - k)))
        if h[best] < h[k]:
            return best
    return k

def pour_argmin(heights, volume, k):
    h = heights[:]
    for _ in range(volume):
        h[settle(h, k)] += 1
    return h

for heights, volume, k, expected in examples:
    got = pour_argmin(heights, volume, k)
    assert got == expected, (got, expected)
    print(heights, volume, k, "->", got)

[2, 1, 1, 2, 1, 2, 2] 4 3 -> [2, 2, 2, 3, 2, 2, 2]
[1, 2, 3, 4] 2 2 -> [2, 3, 3, 4]
[3, 1, 3] 5 1 -> [4, 4, 4]


In [14]:
# Code nộp LeetCode — Cách 3: min theo khóa (mức, khoảng cách)
class Solution:
    def pourWater(self, heights: List[int], volume: int, k: int) -> List[int]:
        n = len(heights)

        def settle(h, k):
            for d in (-1, 1):
                run = [k]
                i = k
                while 0 <= i + d < n and h[i + d] <= h[i]:
                    i += d
                    run.append(i)
                best = min(run, key=lambda x: (h[x], abs(x - k)))
                if h[best] < h[k]:
                    return best
            return k

        for _ in range(volume):
            heights[settle(heights, k)] += 1
        return heights

for heights, volume, k, expected in examples:
    assert Solution().pourWater(list(heights), volume, k) == expected
print("Cách 3 qua ví dụ đề")

Cách 3 qua ví dụ đề


**Độ phức tạp.** **Time O(v · n)** (mỗi giọt dựng đoạn chảy và `min` đều O(n)), **Space O(n)** cho danh sách `run` (tạm thời mỗi giọt). Dễ đọc hơn cách 2 nhưng tốn thêm bộ nhớ nhỏ.

### 7.3. Cách 4 — Đổ theo lô: gom nhiều giọt cùng rơi vào một ô

**Ý tưởng.** Khi `volume = 2000` mà `n = 1` thì ta tính lại điểm dừng 2000 lần dù nó luôn là cột `k`. Ta nhận xét: sau khi tìm được ô dừng `t`, **nhiều giọt liên tiếp** sẽ cùng dừng ở `t` cho đến khi mức của `t` **chạm** mức một ô kề với nó (lúc đó luồng chảy có thể đổi).

Cụ thể, với ô dừng `t` và mỗi ô kề `nb` (`t - 1`, `t + 1`, nếu còn trong mảng):

- Nếu `h[nb] > h[t]` thì có thể đổ tối đa `h[nb] - h[t]` giọt vào `t` mà `t` vẫn **thấp hơn hẳn** `nb` (đường đi tới `t` không đổi).
- Nếu `h[nb] <= h[t]` (mặt bằng bằng nhau) thì chỉ an toàn đổ **1** giọt (vì sau đó `t` cao hơn `nb` và giọt kế tiếp sẽ chảy sang `nb`).

Lô = `min(còn lại, mọi giới hạn trên)` (ít nhất `1`). Đổ cả lô một lần rồi tính lại `settle`. Cách này dùng hàm `settle` của cách 3.

**Vì sao đúng (tóm tắt):** trong suốt lô, đường đi của giọt từ `k` tới `t` không thay đổi (các ô khác không đổi, `t` luôn thấp hơn hẳn ô trước nó trên đường), và `t` vẫn là ô dừng vì ô ngay sau nó (nếu có) còn **cao hơn hẳn** `t` (nếu chỉ cần đạt mức bằng thì lô đã dừng, giới hạn `h[nb] - h[t]`).

In [15]:
def pour_batch(heights, volume, k):
    h = heights[:]
    n = len(h)
    rem = volume
    while rem > 0:
        t = settle(h, k)
        cap = rem
        for nb in (t - 1, t + 1):
            if 0 <= nb < n:
                diff = h[nb] - h[t]
                cap = min(cap, diff if diff > 0 else 1)
        h[t] += cap
        rem -= cap
    return h

for heights, volume, k, expected in examples:
    got = pour_batch(heights, volume, k)
    assert got == expected, (got, expected)
    print(heights, volume, k, "->", got)

[2, 1, 1, 2, 1, 2, 2] 4 3 -> [2, 2, 2, 3, 2, 2, 2]
[1, 2, 3, 4] 2 2 -> [2, 3, 3, 4]
[3, 1, 3] 5 1 -> [4, 4, 4]


In [16]:
# Code nộp LeetCode — Cách 4: đổ theo lô
class Solution:
    def pourWater(self, heights: List[int], volume: int, k: int) -> List[int]:
        n = len(heights)

        def settle():
            for d in (-1, 1):
                i = j = k
                while 0 <= i + d < n and heights[i + d] <= heights[i]:
                    if heights[i + d] < heights[i]:
                        j = i + d
                    i += d
                if j != k:
                    return j
            return k

        rem = volume
        while rem > 0:
            t = settle()
            cap = rem
            for nb in (t - 1, t + 1):
                if 0 <= nb < n:
                    diff = heights[nb] - heights[t]
                    cap = min(cap, diff if diff > 0 else 1)
            heights[t] += cap
            rem -= cap
        return heights

for heights, volume, k, expected in examples:
    assert Solution().pourWater(list(heights), volume, k) == expected
print("Cách 4 qua ví dụ đề")

Cách 4 qua ví dụ đề


**Độ phức tạp.** Mỗi lô tốn một lần `settle` O(n). Số lô ≤ `v` (tệ nhất lô nào cũng chỉ 1 giọt, ví dụ nhiều mặt bằng bằng nhau), nên **Time O(n · số lô) ≤ O(v · n)**. Trường hợp thuận lợi (cột lõm sâu, `n` nhỏ, `volume` lớn) số lô rất nhỏ. **Space O(1)**. Cách này **tinh vi hơn cần thiết** cho giới hạn của bài, nhưng cho thấy ta có thể **gom nhiều bước mô phỏng giống nhau**; ta kiểm chứng nó bằng test mục 12.

## 8. TRACE từng bước

Trace **cách 2** với `heights = [2,1,1,2,1,2,2]`, `volume = 4`, `k = 3`:

| giọt | h trước | hướng trái: đường đi / j | hướng phải: đường đi / j | hành động | h sau |
|---|---|---|---|---|---|
| 1 | [2,1,1,2,1,2,2] | 3→2→1 (mức 2,1,1), tụt tại ô 2: `j = 2` | (không thử vì trái tụt) | dâng ở cột 2 | [2,1,**2**,2,1,2,2] |
| 2 | [2,1,2,2,1,2,2] | 3→2→1 (mức 2,2,1), tụt tại ô 1: `j = 1` | — | dâng ở cột 1 | [2,**2**,2,2,1,2,2] |
| 3 | [2,2,2,2,1,2,2] | 3→2→1→0 (mức 2,2,2,2) không tụt: `j = 3 = k` | 3→4 (mức 2,1) tụt: `j = 4` | dâng ở cột 4 | [2,2,2,2,**2**,2,2] |
| 4 | [2,2,2,2,2,2,2] | không tụt | không tụt | `else`: dâng ở `k = 3` | [2,2,2,**3**,2,2,2] |

Kết quả `[2,2,2,3,2,2,2]` khớp đề. Hãy để hàm in từng bước.

In [17]:
def trace_pour(heights, volume, k):
    h = heights[:]
    n = len(h)
    print("ban đầu:", h, "| k =", k)
    for drop in range(1, volume + 1):
        print(f"--- giọt {drop}: h = {h}")
        landed = None
        for d in (-1, 1):
            name = "trái" if d == -1 else "phải"
            i = j = k
            path = [k]
            while 0 <= i + d < n and h[i + d] <= h[i]:
                if h[i + d] < h[i]:
                    j = i + d
                i += d
                path.append(i)
            print(f"  hướng {name}: đường đi {path} (mức {[h[x] for x in path]}) -> j = {j}", "TỤT" if j != k else "không tụt")
            if j != k:
                landed = j
                break
        if landed is None:
            landed = k
            print("  cả hai hướng không tụt -> dâng tại k")
        h[landed] += 1
        print(f"  => dâng ở cột {landed}, h = {h}")
    return h

assert trace_pour([2, 1, 1, 2, 1, 2, 2], 4, 3) == [2, 2, 2, 3, 2, 2, 2]

ban đầu: [2, 1, 1, 2, 1, 2, 2] | k = 3
--- giọt 1: h = [2, 1, 1, 2, 1, 2, 2]
  hướng trái: đường đi [3, 2, 1] (mức [2, 1, 1]) -> j = 2 TỤT
  => dâng ở cột 2, h = [2, 1, 2, 2, 1, 2, 2]
--- giọt 2: h = [2, 1, 2, 2, 1, 2, 2]
  hướng trái: đường đi [3, 2, 1] (mức [2, 2, 1]) -> j = 1 TỤT
  => dâng ở cột 1, h = [2, 2, 2, 2, 1, 2, 2]
--- giọt 3: h = [2, 2, 2, 2, 1, 2, 2]
  hướng trái: đường đi [3, 2, 1, 0] (mức [2, 2, 2, 2]) -> j = 3 không tụt
  hướng phải: đường đi [3, 4] (mức [2, 1]) -> j = 4 TỤT
  => dâng ở cột 4, h = [2, 2, 2, 2, 2, 2, 2]
--- giọt 4: h = [2, 2, 2, 2, 2, 2, 2]
  hướng trái: đường đi [3, 2, 1, 0] (mức [2, 2, 2, 2]) -> j = 3 không tụt
  hướng phải: đường đi [3, 4, 5, 6] (mức [2, 2, 2, 2]) -> j = 3 không tụt
  cả hai hướng không tụt -> dâng tại k
  => dâng ở cột 3, h = [2, 2, 2, 3, 2, 2, 2]


Trace **ví dụ 2** `heights = [1,2,3,4]`, `volume = 2`, `k = 2` — một ca mà giọt phải đi **xa** hơn một ô và gặp **mặt bằng bằng nhau**:

| giọt | h trước | hướng trái | hành động | h sau |
|---|---|---|---|---|
| 1 | [1,2,3,4] | 2→1→0 (mức 3,2,1): tụt ở ô 1, rồi ô 0: `j = 0` | dâng ở cột 0 | [**2**,2,3,4] |
| 2 | [2,2,3,4] | 2→1→0 (mức 3,2,2): tụt ở ô 1: `j = 1`; ô 0 bằng ô 1, không tụt thêm | dâng ở cột 1 (ô **gần k** nhất) | [2,**3**,3,4] |

Giọt thứ hai chính là chỗ dễ sai nhất: nếu chọn ô xa (cột 0) ta được `[3,2,3,4]` — sai.

In [18]:
assert trace_pour([1, 2, 3, 4], 2, 2) == [2, 3, 3, 4]

ban đầu: [1, 2, 3, 4] | k = 2
--- giọt 1: h = [1, 2, 3, 4]
  hướng trái: đường đi [2, 1, 0] (mức [3, 2, 1]) -> j = 0 TỤT
  => dâng ở cột 0, h = [2, 2, 3, 4]
--- giọt 2: h = [2, 2, 3, 4]
  hướng trái: đường đi [2, 1, 0] (mức [3, 2, 2]) -> j = 1 TỤT
  => dâng ở cột 1, h = [2, 3, 3, 4]


Cuối cùng, trace **cách 4 (đổ theo lô)** với `heights = [3,1,3]`, `volume = 5`, `k = 1`. Theo lập luận ở mục 7.3, các lô là `2, 1, 1, 1` giọt:

In [19]:
def trace_batch(heights, volume, k):
    h = heights[:]
    n = len(h)
    rem = volume
    print("ban đầu:", h)
    step = 0
    while rem > 0:
        step += 1
        t = settle(h, k)
        limits = []
        cap = rem
        for nb in (t - 1, t + 1):
            if 0 <= nb < n:
                diff = h[nb] - h[t]
                lim = diff if diff > 0 else 1
                limits.append((nb, lim))
                cap = min(cap, lim)
        h[t] += cap
        rem -= cap
        print(f"lô {step}: dừng ở {t}, giới hạn theo ô kề {limits} -> đổ {cap} giọt | h = {h} | còn {rem}")
    return h

assert trace_batch([3, 1, 3], 5, 1) == [4, 4, 4]

ban đầu: [3, 1, 3]
lô 1: dừng ở 1, giới hạn theo ô kề [(0, 2), (2, 2)] -> đổ 2 giọt | h = [3, 3, 3] | còn 3
lô 2: dừng ở 1, giới hạn theo ô kề [(0, 1), (2, 1)] -> đổ 1 giọt | h = [3, 4, 3] | còn 2
lô 3: dừng ở 0, giới hạn theo ô kề [(1, 1)] -> đổ 1 giọt | h = [4, 4, 3] | còn 1
lô 4: dừng ở 2, giới hạn theo ô kề [(1, 1)] -> đổ 1 giọt | h = [4, 4, 4] | còn 0


## 9. Code nộp LeetCode

Ta nộp **cách 2** (một lượt quét mỗi giọt): ngắn, dễ chứng minh, `O(v · n)` đủ nhanh với `v ≤ 2000`, `n ≤ 100`.

In [20]:
class Solution:
    def pourWater(self, heights: List[int], volume: int, k: int) -> List[int]:
        n = len(heights)
        for _ in range(volume):
            for d in (-1, 1):
                i = j = k
                while 0 <= i + d < n and heights[i + d] <= heights[i]:
                    if heights[i + d] < heights[i]:
                        j = i + d
                    i += d
                if j != k:
                    heights[j] += 1
                    break
            else:
                heights[k] += 1
        return heights


sol = Solution()
for heights, volume, k, expected in examples:
    got = sol.pourWater(list(heights), volume, k)
    print(heights, volume, k, "->", got, "(kỳ vọng", expected, ")")
    assert got == expected

[2, 1, 1, 2, 1, 2, 2] 4 3 -> [2, 2, 2, 3, 2, 2, 2] (kỳ vọng [2, 2, 2, 3, 2, 2, 2] )
[1, 2, 3, 4] 2 2 -> [2, 3, 3, 4] (kỳ vọng [2, 3, 3, 4] )
[3, 1, 3] 5 1 -> [4, 4, 4] (kỳ vọng [4, 4, 4] )


## 10. Vì sao đúng

**Đoạn chảy.** Với hướng `d`, các ô giọt có thể đi tới là `k = i_0, i_1, ..., i_m` với `h[i_{t+1}] <= h[i_t]` (mức **không tăng**) và `i_{m} + d` hoặc ngoài mảng hoặc cao hơn `h[i_m]`. Vì mức không tăng:

- Mức thấp nhất của đoạn là `h[i_m]` (ô cuối).
- Giọt "cuối cùng tụt xuống khi đi hướng này" ⇔ tồn tại `t` với `h[i_t] < h[k]` ⇔ `h[i_m] < h[k]` ⇔ có **ít nhất một lần tụt thấp hơn hẳn** `h[i_{t+1}] < h[i_t]` trên đường đi. Vòng `while` của cách 2 ghi nhận đúng điều đó qua `j`.

**Ô dừng.** Trong luật của đề, giọt tiếp tục lăn cho đến khi không còn tụt được nữa; trên đoạn chảy, nó dừng ở ô đầu tiên (từ `k`) có mức thấp nhất của đoạn: đến đó thì phía trước chỉ còn các ô **bằng** (không tụt thêm) hoặc ô cao hơn / hết mảng. Ô đó chính là ô được gán cho `j` ở **lần tụt thấp hơn hẳn cuối cùng** (các ô bằng nhau sau đó không đổi `j`). Cũng bằng cách đó cách 3 chọn `min` theo `(mức, khoảng cách)`: trong đoạn chảy không tăng, các ô mức thấp nhất liền kề nhau ở cuối đoạn, ô gần `k` nhất là ô đầu của mặt bằng.

**Thứ tự ưu tiên.** Vòng `for d in (-1, 1)` thử trái trước; chỉ khi trái không tụt mới tới phải; nếu cả hai không tụt thì `else` dâng tại `k` — khớp đúng ba luật của đề.

**Biên.** Điều kiện `0 <= i + d < n` biến hai đầu mảng thành "tường vô hạn": giọt không bao giờ đi ra ngoài, và không có ô "thấp hơn" ngoài biên để tụt xuống.

**Cách 1** đúng vì bám sát đề: tại mỗi vị trí hỏi lại luật. **Cách 4** đúng nhờ lập luận ở mục 7.3: trong một lô, đường đi và ô dừng không đổi, chỉ có mức của `t` tăng đều một đơn vị mỗi giọt; lô dừng đúng trước khi mức `t` bằng mức một ô kề (khi đó luồng có thể đổi). Test ở mục 12 kiểm chứng điều này trên hàng nghìn ca ngẫu nhiên.

## 11. Time / Space Complexity

Gọi `v` là `volume`, `n` là độ dài mảng.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Giọt lăn từng bước | O(v · n²) | O(1) | trung thành đề; làm trọng tài |
| 2. Một lượt quét mỗi giọt | O(v · n) | O(1) | **cách nộp** |
| 3. `min` theo khóa | O(v · n) | O(n) | dễ đọc, hơi tốn bộ nhớ tạm |
| 4. Đổ theo lô | O(n · số lô) ≤ O(v · n) | O(1) | nhanh khi `volume` lớn mà ít "sự kiện" |

Với `n ≤ 100`, `v ≤ 2000`: cách 2 chỉ vài trăm nghìn thao tác.

## 12. Bộ test

Vì ba cách (2, 3, 4) và trọng tài (cách 1) có thể **cùng hiểu sai đề** theo một kiểu, ta kết hợp: ví dụ đề, **ca tính tay**, **tính chất bất biến** (bảo toàn nước, không thấp hơn địa hình), rồi mới so các cách với nhau trên ngẫu nhiên.

In [21]:
all_ways = {
    "pour_ball": pour_ball,
    "pour_one_pass": pour_one_pass,
    "pour_argmin": pour_argmin,
    "pour_batch": pour_batch,
}

# ca tính tay (tự dựng, đã kiểm tay theo luật)
hand = [
    ([0, 0, 0], 1, 1, [0, 1, 0]),          # phẳng: dâng tại k
    ([0, 0, 0], 2, 1, [1, 1, 0]),          # giọt 2: tụt sang trái
    ([0, 0, 0], 3, 1, [1, 1, 1]),          # giọt 3: trái không tụt, phải tụt
    ([0, 0, 0], 4, 1, [1, 2, 1]),
    ([5], 7, 0, [12]),                     # một cột: dâng mãi
    ([4, 3, 2, 1], 3, 0, [4, 3, 3, 3]),    # dốc xuống, k ở đỉnh
    ([3, 1, 2], 2, 1, [3, 3, 2]),
    ([1, 2, 3, 4], 0, 2, [1, 2, 3, 4]),    # volume = 0
    ([0, 1], 1, 1, [1, 1]),               # trái thấp hơn: tụt sang trái
]
tests = examples + hand
for heights, volume, k, expected in tests:
    for name, fn in all_ways.items():
        got = fn(list(heights), volume, k)
        assert got == expected, (name, heights, volume, k, got, expected)
print("OK:", len(tests), "ca cố định (ví dụ đề + tính tay), 4 cách khớp")

OK: 12 ca cố định (ví dụ đề + tính tay), 4 cách khớp


Hãy tự kiểm tra bằng tay một ca để chắc hiểu: `[4,3,2,1]`, `volume = 3`, `k = 0`. Giọt 1: đi trái ra ngoài mảng (tường) không được; đi phải: `3<4`, `2<3`, `1<2` → `j = 3` → `[4,3,2,2]`. Giọt 2: đi phải `3<4 (j=1)`, `2<3 (j=2)`, `2<=2` (bằng, đi tiếp nhưng không đổi `j`) → `j = 2` → `[4,3,3,2]`. Giọt 3: `3<4 (j=1)`, `3<=3` (bằng), `2<3 (j=3)` → `j = 3` → `[4,3,3,3]`. Đúng như bảng trên.

### Tính chất bất biến

Với mọi đầu vào hợp lệ, kết quả phải thỏa:

1. **Bảo toàn nước:** `sum(kết quả) == sum(heights) + volume`.
2. **Không chìm dưới địa hình:** `kết quả[i] >= heights[i]` với mọi `i`.
3. **Cột `k` nhận nước sau cùng:** nếu `volume > 0` thì kết quả khác `heights`.

In [22]:
def check_invariants(heights, volume, k, result):
    assert sum(result) == sum(heights) + volume
    assert all(r >= x for r, x in zip(result, heights))
    if volume > 0:
        assert result != heights

random.seed(755)
for _ in range(500):
    n = random.randint(1, 8)
    heights = [random.randint(0, 5) for _ in range(n)]
    volume = random.randint(0, 20)
    k = random.randrange(n)
    for name, fn in all_ways.items():
        check_invariants(heights, volume, k, fn(list(heights), volume, k))
print("OK: các tính chất bất biến trên 500 ca ngẫu nhiên")

OK: các tính chất bất biến trên 500 ca ngẫu nhiên


### Test ngẫu nhiên nhỏ: so 4 cách với nhau

`n` từ 1 đến 10, độ cao `0..6` (nhiều mặt bằng bằng nhau), `volume` đến 40. Seed cố định. Cách 1 (bám đề) làm tham chiếu.

In [23]:
random.seed(7550)
for _ in range(3000):
    n = random.randint(1, 10)
    heights = [random.randint(0, 6) for _ in range(n)]
    volume = random.randint(0, 40)
    k = random.randrange(n)
    ref = pour_ball(list(heights), volume, k)
    for name, fn in all_ways.items():
        got = fn(list(heights), volume, k)
        assert got == ref, (name, heights, volume, k, got, ref)
print("OK: 3000 ca ngẫu nhiên nhỏ, 4 cách khớp nhau")

OK: 3000 ca ngẫu nhiên nhỏ, 4 cách khớp nhau


### Test ngẫu nhiên đúng ràng buộc (n tới 100, volume tới 2000)

Cách 1 quá chậm ở cỡ này nên so cách 2, 3, 4 với nhau, và so cả ba với cách 1 trên vài ca vừa. Độ cao `0..99`, nhưng cũng thử độ cao hẹp `0..3` (nhiều mặt bằng) và địa hình dạng "chảo".

In [24]:
random.seed(75500)
for t in range(300):
    n = random.randint(1, 100)
    hi = random.choice([3, 10, 99])
    heights = [random.randint(0, hi) for _ in range(n)]
    volume = random.randint(0, 2000)
    k = random.randrange(n)
    ref = pour_one_pass(list(heights), volume, k)
    assert pour_argmin(list(heights), volume, k) == ref
    assert pour_batch(list(heights), volume, k) == ref
    check_invariants(heights, volume, k, ref)
print("OK: 300 ca cỡ đề, cách 2/3/4 khớp nhau")

random.seed(755001)
for t in range(40):
    n = random.randint(1, 25)
    heights = [random.randint(0, 9) for _ in range(n)]
    volume = random.randint(0, 150)
    k = random.randrange(n)
    assert pour_ball(list(heights), volume, k) == pour_one_pass(list(heights), volume, k)
print("OK: 40 ca vừa, cách 1 khớp cách 2")

OK: 300 ca cỡ đề, cách 2/3/4 khớp nhau
OK: 40 ca vừa, cách 1 khớp cách 2


### Ca đặc biệt cỡ tối đa

- **Chảo** `[99, 0, 0, ..., 0, 99]` với `volume = 2000`: nước lấp đầy lòng chảo, phần dư `dâng` — kiểm tra bằng bảo toàn.
- **Dốc thoải** tăng/giảm đều.
- **Một cột**: cả `volume` dâng lên một cột.

In [25]:
n = 100
bowl = [99] + [0] * (n - 2) + [99]
dec = list(range(99, -1, -1))
inc = list(range(100))
single = [7]
cases = [(bowl, 2000, 50), (bowl, 2000, 0), (dec, 2000, 0), (dec, 2000, 99), (inc, 2000, 99), (inc, 2000, 0), (single, 2000, 0)]
for heights, volume, k in cases:
    ref = pour_one_pass(list(heights), volume, k)
    assert pour_batch(list(heights), volume, k) == ref
    assert pour_argmin(list(heights), volume, k) == ref
    check_invariants(heights, volume, k, ref)
print("OK: các ca đặc biệt cỡ tối đa")
res_bowl = pour_one_pass(list(bowl), 2000, 50)
print("chảo, v=2000, k=50 -> mức thấp nhất trong lòng chảo:", min(res_bowl[1:-1]), "| cao nhất:", max(res_bowl[1:-1]), "| tổng:", sum(res_bowl))
assert max(res_bowl[1:-1]) - min(res_bowl[1:-1]) <= 1
assert pour_one_pass(list(single), 2000, 0) == [2007]

OK: các ca đặc biệt cỡ tối đa
chảo, v=2000, k=50 -> mức thấp nhất trong lòng chảo: 20 | cao nhất: 21 | tổng: 2198


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dừng ngay khi gặp ô bằng nhau (dùng `<` thay `<=` khi đi)

Nếu điều kiện đi tiếp là `h[i + d] < h[i]` (chỉ đi xuống hẳn), giọt **không vượt qua mặt bằng** để tới chỗ trũng phía sau. Ví dụ `[1,2,2,2]` với `k = 3`: đúng là giọt đi qua hai ô mức `2` (bằng nhau) rồi tụt xuống ô mức `1` ở cột `0`.

In [26]:
def wrong_strict_walk(heights, volume, k):
    h = heights[:]
    n = len(h)
    for _ in range(volume):
        for d in (-1, 1):
            i = j = k
            while 0 <= i + d < n and h[i + d] < h[i]:      # SAI: phải là <=
                j = i + d
                i += d
            if j != k:
                h[j] += 1
                break
        else:
            h[k] += 1
    return h

heights, volume, k = [1, 2, 2, 2], 1, 3
print("sai    :", wrong_strict_walk(heights, volume, k), "| đúng:", pour_one_pass(heights, volume, k))
assert wrong_strict_walk(heights, volume, k) == [1, 2, 2, 3]
assert pour_one_pass(heights, volume, k) == [2, 2, 2, 2]

sai    : [1, 2, 2, 3] | đúng: [2, 2, 2, 2]


**Vì sao sai?** Giọt ở cột `3` (mức 2), cột kề mức `2` bằng nhau: theo đề nó **vẫn đi được** (mức "bằng hoặc thấp hơn"), và cuối cùng gặp cột `0` mức `1` → **tụt**. Code sai dừng ngay vì `2 < 2` sai, nên dâng tại `k`. **Sửa:** đi bằng `<=`, chỉ ghi `j` khi `<`.

### Lỗi 2 — Cập nhật `j` cả khi bằng nhau (chọn ô xa `k` nhất)

Nếu viết `if h[i + d] <= h[i]: j = i + d` thì `j` bị kéo ra **xa** `k` trên mặt bằng bằng nhau; đề cần ô **gần `k` nhất** (nơi lần tụt hẳn cuối cùng).

In [27]:
def wrong_far(heights, volume, k):
    h = heights[:]
    n = len(h)
    for _ in range(volume):
        for d in (-1, 1):
            i = j = k
            while 0 <= i + d < n and h[i + d] <= h[i]:
                if h[i + d] <= h[i] and h[i + d] < h[k]:   # SAI: gán j cả khi bằng ô trước
                    j = i + d
                i += d
            if j != k:
                h[j] += 1
                break
        else:
            h[k] += 1
    return h

heights, volume, k = [1, 2, 3, 4], 2, 2
print("sai    :", wrong_far(heights, volume, k), "| đúng:", pour_one_pass(heights, volume, k))
assert wrong_far(heights, volume, k) == [3, 2, 3, 4]
assert pour_one_pass(heights, volume, k) == [2, 3, 3, 4]

sai    : [3, 2, 3, 4] | đúng: [2, 3, 3, 4]


**Vì sao sai?** Giọt thứ hai (ví dụ 2) thấy ô `1` và ô `0` cùng mức `2`; code sai chọn ô xa (cột `0`) thay vì cột `1`. **Sửa:** chỉ `j = i + d` khi `h[i + d] < h[i]` (thấp hơn hẳn ô **ngay trước nó**).

### Lỗi 3 — Quên ưu tiên **trái trước phải**

Nếu thử phải trước (hoặc chọn hướng tụt **sâu hơn**), kết quả đổi. Đề ghi rõ: nếu trái tụt thì đi trái, **kể cả khi phải tụt sâu hơn**.

In [28]:
def wrong_right_first(heights, volume, k):
    h = heights[:]
    n = len(h)
    for _ in range(volume):
        for d in (1, -1):                                   # SAI: phải trước
            i = j = k
            while 0 <= i + d < n and h[i + d] <= h[i]:
                if h[i + d] < h[i]:
                    j = i + d
                i += d
            if j != k:
                h[j] += 1
                break
        else:
            h[k] += 1
    return h

heights, volume, k = [2, 1, 1, 2, 1, 2, 2], 2, 3
print("sai    :", wrong_right_first(heights, volume, k), "| đúng:", pour_one_pass(heights, volume, k))
assert wrong_right_first(heights, volume, k) != pour_one_pass(heights, volume, k)
assert pour_one_pass(heights, volume, k) == [2, 2, 2, 2, 1, 2, 2]

sai    : [2, 1, 2, 2, 2, 2, 2] | đúng: [2, 2, 2, 2, 1, 2, 2]


### Lỗi 4 — Không coi biên là tường (đọc `h[-1]` hoặc ra ngoài mảng)

Nếu không kiểm tra `0 <= i + d < n`, `h[i + d]` với `i + d = -1` trong Python **không báo lỗi**: nó đọc phần tử **cuối** của mảng (chỉ số âm!), làm nước "chảy vòng" sang đầu kia. Còn `i + d = n` thì `IndexError`.

In [29]:
h = [5, 3, 1]
print("h[-1] =", h[-1], "(đọc phần tử cuối thay vì lỗi)")

def wrong_no_bounds(heights, volume, k):
    h = heights[:]
    n = len(h)
    for _ in range(volume):
        for d in (-1, 1):
            i = j = k
            steps = 0
            while steps < n and h[(i + d) % n] <= h[i]:     # SAI: coi mảng là vòng tròn
                if h[(i + d) % n] < h[i]:
                    j = (i + d) % n
                i = (i + d) % n
                steps += 1
            if j != k:
                h[j] += 1
                break
        else:
            h[k] += 1
    return h

heights, volume, k = [3, 5, 5, 1], 1, 1
print("sai    :", wrong_no_bounds(heights, volume, k), "| đúng:", pour_one_pass(heights, volume, k))
assert wrong_no_bounds(heights, volume, k) != pour_one_pass(heights, volume, k)

h[-1] = 1 (đọc phần tử cuối thay vì lỗi)
sai    : [3, 5, 5, 2] | đúng: [4, 5, 5, 1]


**Sửa:** luôn viết `0 <= i + d < n` **trước** `h[i + d] <= h[i]` trong điều kiện `while` (thứ tự quan trọng nhờ ngắn mạch).

### Lỗi 5 — Sửa đầu vào ngoài ý muốn khi viết nhiều cách

Các hàm `pour_*` ở trên đều bắt đầu bằng `h = heights[:]` để **không** làm hỏng đầu vào. Nếu quên và sửa trực tiếp `heights`, lần so sánh cách thứ hai sẽ chạy trên dữ liệu **đã bị đổi**.

In [30]:
def wrong_mutates(heights, volume, k):
    n = len(heights)
    for _ in range(volume):
        for d in (-1, 1):
            i = j = k
            while 0 <= i + d < n and heights[i + d] <= heights[i]:
                if heights[i + d] < heights[i]:
                    j = i + d
                i += d
            if j != k:
                heights[j] += 1
                break
        else:
            heights[k] += 1
    return heights

data = [2, 1, 1, 2, 1, 2, 2]
first = wrong_mutates(data, 4, 3)[:]     # sao chép kết quả lần 1 để so sánh
second = wrong_mutates(data, 4, 3)      # data đã bị đổi ở lần gọi đầu
print("lần 1:", first)
print("lần 2:", second, "<- không còn là kết quả đúng của đầu vào gốc")
assert first == [2, 2, 2, 3, 2, 2, 2] and second != first

lần 1: [2, 2, 2, 3, 2, 2, 2]
lần 2: [3, 3, 3, 3, 3, 2, 2] <- không còn là kết quả đúng của đầu vào gốc


Trên LeetCode việc sửa tại chỗ `heights` rồi trả về chính nó được chấp nhận (mỗi test có đầu vào mới), nhưng khi **tự kiểm thử** nhiều cách trên cùng dữ liệu hãy truyền `list(heights)`.

## 14. Mẹo & nhận dạng pattern

- **Bài "mô phỏng quá trình theo luật"** với giới hạn nhỏ (`n ≤ 100`, `v ≤ 2000`) → cứ **mô phỏng thẳng** và tối ưu việc xử lý **một bước**; đừng tìm công thức.
- **Hai hướng đối xứng** → `for d in (-1, 1)`; thứ tự trong tuple quyết định ưu tiên (trái trước).
- **"Đi khi không bị chặn, nhớ vị trí tốt nhất"** → một con trỏ `i` đi, một biến `j` ghi ứng viên; cập nhật `j` **chỉ khi có cải thiện hẳn** (thấp hơn hẳn) để tie-break "gần nhất" tự nhiên đúng.
- **Mặt bằng bằng nhau** là nơi mọi bài kiểu này sai: tự hỏi "bằng nhau thì đi tiếp hay dừng? có tính là tụt không?".
- **Biên = tường vô hạn** → điều kiện biên trong `while` kèm `and` ngắn mạch.
- **Kiểm thử bài mô phỏng:** vẽ bản đồ bằng chữ (`draw`), có trọng tài bám đề, **tính chất bất biến** (bảo toàn tổng), và ca ngẫu nhiên có nhiều giá trị bằng nhau.
- **Gom bước giống nhau (batch)**: nếu thấy nhiều lần lặp cùng một quyết định, tìm điều kiện "khi nào quyết định đổi" để nhảy một lèo.
- Họ hàng: *Trapping Rain Water* (42, 407) — nước chứa trong chảo; bài này khác ở chỗ **nước rơi từng giọt có luật chảy**.

## 15. Biến thể

### 15.1. Trả về nơi **từng giọt** dừng lại

Chỉ cần ghi lại `settle(h, k)` mỗi lần trước khi đổ.

In [31]:
def landing_sites(heights, volume, k):
    h = heights[:]
    sites = []
    for _ in range(volume):
        t = settle(h, k)
        sites.append(t)
        h[t] += 1
    return sites

print(landing_sites([2, 1, 1, 2, 1, 2, 2], 4, 3))      # [2, 1, 4, 3]
assert landing_sites([2, 1, 1, 2, 1, 2, 2], 4, 3) == [2, 1, 4, 3]
assert landing_sites([1, 2, 3, 4], 2, 2) == [0, 1]

[2, 1, 4, 3]


### 15.2. Ưu tiên **phải** trước (đổi luật)

Chỉ cần đảo tuple hướng: `for d in (1, -1)`. Kiểm tra bằng tính đối xứng: đổ ở ảnh gương rồi lật lại phải cho cùng kết quả với luật ưu tiên-phải.

In [32]:
def pour_right_first(heights, volume, k):
    h = heights[:]
    n = len(h)
    for _ in range(volume):
        for d in (1, -1):
            i = j = k
            while 0 <= i + d < n and h[i + d] <= h[i]:
                if h[i + d] < h[i]:
                    j = i + d
                i += d
            if j != k:
                h[j] += 1
                break
        else:
            h[k] += 1
    return h

random.seed(152)
for _ in range(300):
    n = random.randint(1, 9)
    heights = [random.randint(0, 5) for _ in range(n)]
    volume = random.randint(0, 25)
    k = random.randrange(n)
    mirror = pour_one_pass(heights[::-1], volume, n - 1 - k)[::-1]
    assert pour_right_first(heights, volume, k) == mirror
print("ưu tiên phải = ảnh gương của ưu tiên trái: OK")

ưu tiên phải = ảnh gương của ưu tiên trái: OK


### 15.3. Hai điểm đổ luân phiên

Giả sử nước rơi **xen kẽ** tại `k1`, `k2` (giọt lẻ ở `k1`, giọt chẵn ở `k2`). Chỉ cần thay `k` theo chẵn lẻ của chỉ số giọt.

In [33]:
def pour_two_sources(heights, volume, k1, k2):
    h = heights[:]
    for t in range(volume):
        k = k1 if t % 2 == 0 else k2
        h[settle(h, k)] += 1
    return h

res = pour_two_sources([3, 0, 0, 0, 3], 4, 1, 3)
print(res)
assert sum(res) == 6 + 4
assert pour_two_sources([3, 0, 0, 0, 3], 0, 1, 3) == [3, 0, 0, 0, 3]

[3, 1, 1, 2, 3]


### 15.4. Bài liên quan

- **42. Trapping Rain Water** và **407. Trapping Rain Water II** — nước đọng trong chảo (không mô phỏng từng giọt).
- **11. Container With Most Water** — hai con trỏ trên mảng độ cao.
- Họ "mô phỏng theo luật": **289. Game of Life**, **346/348** (thiết kế), **735. Asteroid Collision**.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Giọt lăn từng bước | hỏi lại luật `falls` tại mỗi ô, bước một ô | O(v·n²) | O(1) | để hiểu đề, làm trọng tài |
| 2. Một lượt quét | đi khi `<=`, ghi `j` khi `<`; trái trước phải | O(v·n) | O(1) | **nên nộp**: gọn, nhanh |
| 3. `min` theo khóa | dựng đoạn chảy, chọn `(mức, khoảng cách)` nhỏ nhất | O(v·n) | O(n) | dễ giải thích với người mới |
| 4. Đổ theo lô | gom các giọt cùng ô dừng đến khi chạm mức ô kề | O(n·số lô) | O(1) | `volume` rất lớn so với `n` |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng — **không được gọi** trước khi bạn viết xong. Đáp án ở mục 19.

### Bài 1 (đoán output)

Không chạy code, hãy đoán `result` rồi điền vào `my_guess_1`.

In [34]:
heights = [3, 1, 2]
result = pour_one_pass(heights, 2, 1)
# print(result)   # <- bỏ dấu # sau khi đã đoán xong để kiểm tra

my_guess_1 = None    # TODO: điền list bạn đoán

### Bài 2 (tính tay)

Với `heights = [4, 3, 2, 1]`, `volume = 3`, `k = 0` (nước rơi ở đầu bên trái, dốc xuống về bên phải): tính tay từng giọt (như mục 12) và điền kết quả vào `my_answer_2`.

In [35]:
my_answer_2 = None    # TODO: điền list kết quả

### Bài 3 (điền TODO)

Hoàn thành hàm một lượt quét mỗi giọt: thay `...` bằng điều kiện / câu lệnh đúng.

In [36]:
def exercise3(heights, volume, k):
    h = heights[:]
    n = len(h)
    for _ in range(volume):
        for d in (-1, 1):
            i = j = k
            while ...:                       # TODO: còn trong mảng và ô kề KHÔNG cao hơn ô hiện tại
                if ...:                      # TODO: ô kề thấp hơn HẲN ô hiện tại
                    j = i + d
                i += d
            if j != k:
                ...                          # TODO: dâng nước ở ô j
                break
        else:
            ...                              # TODO: cả hai hướng không tụt: dâng ở đâu?
    return h

### Bài 4 (tự viết lại)

Viết `exercise4(h, k)` trả về **chỉ số** nơi giọt nước kế tiếp sẽ dừng (không sửa `h`). Dùng bất kỳ cách nào bạn thích (một lượt quét hoặc `min` theo khóa). Ví dụ `exercise4([2,1,1,2,1,2,2], 3)` → `2`.

In [37]:
def exercise4(h, k):
    # TODO: viết lời giải
    pass

### Bài 5 (biến thể / mini challenge)

Viết `exercise5(heights, volume, k)` trả về **list các chỉ số** nơi từng giọt dừng, theo thứ tự rơi (xem mục 15.1). Ví dụ `exercise5([2,1,1,2,1,2,2], 4, 3)` → `[2, 1, 4, 3]`. (Gợi ý: có thể dùng lại ý tưởng của bài 4.)

In [38]:
def exercise5(heights, volume, k):
    # TODO: viết lời giải
    pass

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `heights = [3,1,2]`, `k = 1` (mức 1). Giọt 1: xét trái rồi phải.
- Mức 2: trái: ô kề mức `3` cao hơn → không đi được. Phải: ô kề mức `2` cao hơn → không đi được. Cả hai không tụt → dâng tại `k`.
- Mức 3: sau giọt 1 ta có `[3,2,2]`. Giọt 2 tại `k` mức `2`: trái `3 > 2` không đi; phải `2 <= 2` đi được nhưng không có ô thấp hơn → không tụt; dâng tại `k` → `[3,3,2]`.

**Bài 2**
- Mức 1: `k = 0`, trái là tường vô hạn nên chỉ có hướng phải.
- Mức 2: giọt 1 tụt tới cột 3 (mức 1 thấp nhất). Giọt 2: gặp ô `2` rồi `2` (bằng), dừng ở ô gần `k` nhất trong các ô mức thấp nhất.
- Mức 3: `[4,3,2,1]` → `[4,3,2,2]` → `[4,3,3,2]` → `[4,3,3,3]`.

**Bài 3**
- Mức 1: nhìn lại mục 7.1.
- Mức 2: điều kiện `while`: `0 <= i + d < n and h[i + d] <= h[i]`; điều kiện `if`: `h[i + d] < h[i]`.
- Mức 3: `h[j] += 1` và `h[k] += 1`.

**Bài 4**
- Mức 1: với mỗi hướng `d`, đi khi ô kề không cao hơn; nhớ ô tụt cuối.
- Mức 2: nếu `j != k` trả `j` ngay (ưu tiên trái trước); nếu cả hai hướng không tụt trả `k`.
- Mức 3: copy cách 2 nhưng thay `h[j] += 1` bằng `return j`, và sau vòng `for` trả `k`.

**Bài 5**
- Mức 1: lặp `volume` lần, mỗi lần gọi bài 4.
- Mức 2: ghi chỉ số vào list rồi **thực sự đổ** giọt (`h[t] += 1`) trên bản sao để trạng thái thay đổi.
- Mức 3: `h = heights[:]`; `t = exercise4(h, k)`; `sites.append(t)`; `h[t] += 1`.

## 19. Đáp án tham khảo

Chạy các cell sau để kiểm tra bằng `assert`.

In [39]:
# Bài 1
answer_1 = pour_one_pass([3, 1, 2], 2, 1)
print("Bài 1:", answer_1)
assert answer_1 == [3, 3, 2]

# Bài 2
answer_2 = pour_ball([4, 3, 2, 1], 3, 0)
print("Bài 2:", answer_2)
assert answer_2 == [4, 3, 3, 3]

Bài 1: [3, 3, 2]
Bài 2: [4, 3, 3, 3]


In [40]:
# Bài 3
def exercise3_solution(heights, volume, k):
    h = heights[:]
    n = len(h)
    for _ in range(volume):
        for d in (-1, 1):
            i = j = k
            while 0 <= i + d < n and h[i + d] <= h[i]:
                if h[i + d] < h[i]:
                    j = i + d
                i += d
            if j != k:
                h[j] += 1
                break
        else:
            h[k] += 1
    return h

# Bài 4
def exercise4_solution(h, k):
    n = len(h)
    for d in (-1, 1):
        i = j = k
        while 0 <= i + d < n and h[i + d] <= h[i]:
            if h[i + d] < h[i]:
                j = i + d
            i += d
        if j != k:
            return j
    return k

# Bài 5
def exercise5_solution(heights, volume, k):
    h = heights[:]
    sites = []
    for _ in range(volume):
        t = exercise4_solution(h, k)
        sites.append(t)
        h[t] += 1
    return sites

for heights, volume, k, expected in examples:
    assert exercise3_solution(heights, volume, k) == expected
assert exercise4_solution([2, 1, 1, 2, 1, 2, 2], 3) == 2
assert exercise4_solution([3, 3, 3], 1) == 1
assert exercise5_solution([2, 1, 1, 2, 1, 2, 2], 4, 3) == [2, 1, 4, 3]
print("Đáp án bài 3, 4, 5 đúng trên ví dụ")

random.seed(19)
for _ in range(500):
    n = random.randint(1, 9)
    heights = [random.randint(0, 5) for _ in range(n)]
    volume = random.randint(0, 25)
    k = random.randrange(n)
    want = pour_ball(list(heights), volume, k)
    assert exercise3_solution(heights, volume, k) == want
    h = heights[:]
    for t in exercise5_solution(heights, volume, k):
        h[t] += 1
    assert h == want
print("Đối chiếu ngẫu nhiên OK")

Đáp án bài 3, 4, 5 đúng trên ví dụ
Đối chiếu ngẫu nhiên OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (mô phỏng từng giọt, O(v·n) thời gian, O(1) bộ nhớ):**

```python
class Solution:
    def pourWater(self, heights: List[int], volume: int, k: int) -> List[int]:
        n = len(heights)
        for _ in range(volume):
            for d in (-1, 1):                      # trái trước, phải sau
                i = j = k
                while 0 <= i + d < n and heights[i + d] <= heights[i]:
                    if heights[i + d] < heights[i]:    # tụt thấp hơn HẲN
                        j = i + d
                    i += d
                if j != k:
                    heights[j] += 1                # dâng ở ô tụt cuối (gần k nhất)
                    break
            else:
                heights[k] += 1                    # cả hai hướng không tụt
        return heights
```

**Checklist trước khi submit:**

- [ ] Đi tiếp khi `<=` (qua mặt bằng bằng nhau), nhưng chỉ ghi `j` khi `<` (thấp hơn hẳn).
- [ ] Thử **trái trước**; chỉ khi trái không tụt mới thử phải (kể cả khi phải tụt sâu hơn).
- [ ] Nếu cả hai hướng không tụt thì dâng tại **`k`**.
- [ ] Điều kiện biên `0 <= i + d < n` đứng **trước** `heights[i + d]` trong `while`.
- [ ] Ô dừng là ô **gần `k` nhất** trong các ô thấp nhất (đúng nhờ chỉ cập nhật `j` khi `<`).
- [ ] Tổng kết quả `== sum(heights) + volume`; `volume = 0` trả nguyên mảng.
- [ ] Đã thử: `[0,0,0]` (phẳng), một cột, dốc xuống, `[1,2,3,4]`, `[3,1,3]` với `volume` lớn.
- [ ] Khi tự kiểm thử nhiều cách, truyền bản sao `list(heights)`.